# GX_08 — Scoring new customers as they arrive (Spark Structured Streaming)

**The question this notebook answers:** *Can the bank get a churn alert for a new customer as soon as the customer's data arrives, instead of waiting for the next monthly batch run?*

### What this notebook is about
GX_04 saved a churn model and GX_06 saved a segmentation model. Both were applied to tables that already existed. In a real bank, customer records keep arriving (new customers, updated balances, new products). Here we show that **the same saved models, the same cleaning and feature code, and the same threshold** can be applied to data that arrives in small pieces, with **Spark Structured Streaming**.

In this notebook we:
1. **Simulate** the arrival of new customer files in a folder (the 1,501 test customers, without their `Exited` value, cut into 10 small files).
2. **Score** every new file as it arrives: features -> segment -> churn probability -> alert if P(churn) >= the GX_04 threshold.
3. **Store** the scored customers in a Delta table (`churn_alerts`), one append per micro-batch.
4. **Prove** that a file is processed exactly once (a second run only reads the new files) and that streaming gives the **same scores** as the batch run in GX_05.
5. **Summarise** the alerts per country and segment, which is what a retention team would read.

### What this notebook uses and produces
| | |
|---|---|
| **Reads** | `gold_churn_features` (only to simulate arriving customers), the churn model from GX_04 and the segmentation model from GX_06 |
| **Writes** | Table `churn_alerts` (one row per scored customer) and temporary landing / checkpoint folders in the `stream` volume |

### Why this is a streaming problem (the 4 V's)
**Velocity** is the V that streaming addresses: new records arrive continuously and the value of an alert drops with time. **Volume** and **Variety** are handled in GX_02 and GX_07. **Veracity** is handled by the quality checks in GX_01.

### Two honest limits
- **The arrival is simulated.** We only have a static file, so we drop its rows into a folder in two rounds. The Spark code is the same code that would read a folder filled by a real system.
- **Free Edition compute is serverless**, which runs streaming queries in *available now* mode: Spark processes everything that is new, then stops. The micro-batch logic, the checkpoint and the exactly-once behaviour are identical to a query that never stops; on a normal cluster only the trigger line changes (`processingTime="1 minute"` instead of `availableNow=True`).

## Step 0 — Preparing the notebook

**What we do:** load the shared helper code and the paths of the two saved models.

**Why:** the streaming job must use **exactly** the same feature code (`build_gold_features`), segmentation code and model as the batch notebooks. If the streaming job had its own copy of the rules, its scores could silently drift away from the validated model.

In [ ]:
# Make the shared helpers in src/ importable (this notebook lives in notebooks/).
import os
import shutil
import sys

REPO_ROOT = os.path.abspath("..")
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

import json

import pandas as pd
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F

from src.config import (
    CHURN_MODEL_PATH, CHURN_MODEL_SETTINGS, GOLD_CHURN, SEGMENT_MODEL_PATH,
    SEGMENT_MODEL_SETTINGS, SILVER_CHURN,
)
from src.features import build_gold_features
from src.ml import KEY_COL, LABEL_COL, SPLIT_COL
from src.segments import SEGMENT_ID_COL, SEGMENT_NAME_COL, add_segment_labels

# Where the "arriving" files, the stream's memory (checkpoint) and the alert table live.
STREAM_VOLUME = "workspace.churn.stream"
STREAM_DIR = "/Volumes/workspace/churn/stream"
LANDING = f"{STREAM_DIR}/landing"
CHECKPOINT = f"{STREAM_DIR}/checkpoint"
ALERTS_TABLE = "workspace.churn.churn_alerts"
N_FILES = 10

spark.sql(f"CREATE VOLUME IF NOT EXISTS {STREAM_VOLUME}")
print("Spark version:", spark.version)

## Step 1 — Load the saved models and start from an empty stream

**What we do:** load the churn model and its threshold (GX_04, Step 12), the segmentation model and its segment names (GX_06, Step 9), and clear any leftovers of an earlier run of this notebook (landing folder, checkpoint, alert table).

**Why clear the leftovers:** a streaming query remembers what it has already read in its *checkpoint* folder. Re-running the demo with an old checkpoint would show "nothing new". Starting clean makes the notebook repeatable.

In [ ]:
churn_model = PipelineModel.load(CHURN_MODEL_PATH)
with open(CHURN_MODEL_SETTINGS) as f:
    THRESHOLD = json.load(f)["threshold"]

segment_model = PipelineModel.load(SEGMENT_MODEL_PATH)
with open(SEGMENT_MODEL_SETTINGS) as f:
    segment_settings = json.load(f)
# cluster number -> (segment_id, segment_name), the format add_segment_labels expects (as in GX_06)
lookup = {int(c): (v["segment_id"], v["segment_name"]) for c, v in segment_settings["segments"].items()}


def reset_demo():
    # Empty landing folder, no checkpoint, no alert table: the stream starts from zero.
    shutil.rmtree(LANDING, ignore_errors=True)
    shutil.rmtree(CHECKPOINT, ignore_errors=True)
    os.makedirs(LANDING, exist_ok=True)
    spark.sql(f"DROP TABLE IF EXISTS {ALERTS_TABLE}")


reset_demo()
print(f"Churn threshold from GX_04: {THRESHOLD}")
print("Segments:", {sid: name for sid, name in lookup.values()})

## Step 2 — Simulating new customers arriving

**What we do:** take the 1,501 test customers, **remove their `Exited` value** (a new customer's future is unknown), sort them by customer number and cut them into 10 CSV files of about 150 customers. A helper function drops files into the landing folder; we will call it twice.

**Why test customers:** they are the customers the model never saw (GX_05 already evaluated it on them and no decision is taken here). Their real `Exited` value is kept apart (`truth`) and only used at the end to check the alerts.

**The columns are the silver customer columns** (`silver_churn` without `Exited`). In the stream, `build_gold_features` creates the same extra columns as in GX_02 (age group, engagement, balance-to-salary, ...) before the model is applied.

**Why an explicit schema:** a streaming file source cannot look at the data first to guess the column types (that would mean reading the files twice), so we give Spark the exact types.

In [ ]:
gold = spark.table(GOLD_CHURN)
incoming_columns = [c for c in spark.table(SILVER_CHURN).columns if c != LABEL_COL]   # no Exited

test_customers = gold.filter(F.col(SPLIT_COL) == "test")
incoming = test_customers.select(*incoming_columns).orderBy(KEY_COL)
truth = test_customers.select(KEY_COL, LABEL_COL)             # kept apart, used only for the final check

stream_schema = incoming.schema                               # exact types for the streaming reader
incoming_pdf = incoming.toPandas()                            # 1,501 rows: tiny, safe on the driver
rows_per_file = -(-len(incoming_pdf) // N_FILES)              # ceiling division
chunks = [incoming_pdf.iloc[i * rows_per_file:(i + 1) * rows_per_file] for i in range(N_FILES)]


def land_files(first, last):
    # Drop files `first`..`last` (1-based) into the landing folder, like a system delivering new data.
    for i in range(first, last + 1):
        chunks[i - 1].to_csv(f"{LANDING}/customers_{i:02d}.csv", index=False)
    n_rows = sum(len(chunks[i - 1]) for i in range(first, last + 1))
    print(f"Landed files {first}-{last} ({n_rows} customers). Folder now holds {len(os.listdir(LANDING))} files.")


print(f"{len(incoming_pdf):,} customers -> {N_FILES} files of up to {rows_per_file} customers")
print("Columns arriving:", incoming_columns)

## Step 3 — The streaming job

**What we do:** define what happens to every micro-batch, and the query that reads the folder.

**The three stages of one micro-batch:**
1. **Features:** `build_gold_features` (the GX_02 code) adds the engineered columns.
2. **Segment:** the saved KMeans pipeline from GX_06 gives each customer a segment (the same order of steps as in GX_06 Step 8).
3. **Churn risk:** the saved MLP pipeline from GX_04 gives P(churn); the customer is an **alert** if P(churn) >= the threshold chosen in GX_04. The result is appended to the Delta table `churn_alerts`.

**How the query works:**
- `readStream` watches the landing folder. `maxFilesPerTrigger = 1` makes every micro-batch read **one file**, so the demo shows several small batches instead of one big one.
- `foreachBatch` hands each micro-batch to our function as an ordinary DataFrame, so the batch code of the other notebooks can be reused unchanged.
- **The checkpoint** is the stream's memory: which files it has already processed. This is what gives **exactly-once** behaviour: after a restart, a file is never scored twice and never skipped.
- `trigger(availableNow=True)` processes everything that is new and then stops (see the note in the introduction).

In [ ]:
def score_batch(batch_df, batch_id):
    # One micro-batch: features -> segment -> churn risk -> append to the alert table.
    if batch_df.isEmpty():
        return
    with_features = build_gold_features(batch_df)
    segmented = add_segment_labels(segment_model.transform(with_features), lookup)
    scored = (
        churn_model.transform(segmented)
        .withColumn("p_churn", vector_to_array("probability")[1].cast("double"))
        .withColumn("alert", (F.col("p_churn") >= THRESHOLD).cast("int"))
    )
    (
        scored.select(KEY_COL, "Geography", "Age", "Balance", "NumOfProducts", "IsActiveMember",
                      SEGMENT_ID_COL, SEGMENT_NAME_COL, "p_churn", "alert")
        .withColumn("batch_id", F.lit(batch_id))
        .withColumn("scored_at", F.current_timestamp())
        .write.mode("append").saveAsTable(ALERTS_TABLE)
    )


def run_stream():
    # Process every file that has not been processed yet, then stop.
    query = (
        spark.readStream
        .schema(stream_schema)
        .option("header", True)
        .option("maxFilesPerTrigger", 1)            # one file per micro-batch
        .csv(LANDING)
        .writeStream
        .foreachBatch(score_batch)
        .option("checkpointLocation", CHECKPOINT)   # the stream's memory: exactly-once
        .trigger(availableNow=True)
        .start()
    )
    query.awaitTermination()
    return query


def show_progress(query):
    # One row per micro-batch: how many customers it read.
    rows = [{"batch": p["batchId"], "customers_read": p["numInputRows"]} for p in query.recentProgress]
    display(pd.DataFrame(rows))

## Step 4 — First arrival: 5 files

**What we do:** drop the first 5 files into the folder and run the stream once.

**What to look for:** one micro-batch per file, each reading about 150 customers, and the alert table filling up batch by batch.

In [ ]:
land_files(1, 5)
query_1 = run_stream()
show_progress(query_1)

display(
    spark.table(ALERTS_TABLE)
    .groupBy("batch_id")
    .agg(F.count("*").alias("customers_scored"), F.sum("alert").alias("alerts"))
    .orderBy("batch_id")
)
print("Customers in the alert table after the first run:", spark.table(ALERTS_TABLE).count())

## Step 5 — Second arrival: 5 more files (exactly-once)

**What we do:** drop the remaining 5 files into the **same folder**, and run the **same query** again, with the same checkpoint.

**What to look for:** the folder now holds 10 files, but the stream must read only the 5 new ones. If it re-read the first 5, the table would contain duplicates. The checkpoint prevents that, and the assertions below fail loudly if it did not.

In [ ]:
land_files(6, 10)
query_2 = run_stream()
show_progress(query_2)

alerts = spark.table(ALERTS_TABLE)
n_rows = alerts.count()
n_distinct = alerts.select(KEY_COL).distinct().count()
print(f"Rows in the alert table: {n_rows:,} | distinct customers: {n_distinct:,} | customers that arrived: {len(incoming_pdf):,}")
assert n_rows == n_distinct == len(incoming_pdf), "A file was processed twice or skipped."
print("Exactly-once confirmed: every customer was scored once, although the folder was read twice.")

# The Delta table keeps one version per micro-batch append.
display(spark.sql(f"DESCRIBE HISTORY {ALERTS_TABLE}").select("version", "timestamp", "operation").orderBy("version"))

## Step 6 — Does streaming give the same scores as the batch run?

**What we do:** score the same 1,501 customers the batch way (the saved model applied to the gold table, as in GX_05) and compare, customer by customer, with the probabilities the stream wrote.

**Why:** a streaming job that scores differently from the validated model would be a different model. The two must agree.

**Then:** we read the real `Exited` values (kept apart in Step 2) to see what the alerts caught. This is the same result as GX_05 and is shown only to confirm that the stream reproduces it.

In [ ]:
batch_scores = (
    churn_model.transform(test_customers)
    .select(KEY_COL, vector_to_array("probability")[1].alias("p_batch"))
)
max_diff = (
    spark.table(ALERTS_TABLE).join(batch_scores, KEY_COL)
    .agg(F.max(F.abs(F.col("p_churn") - F.col("p_batch")))).first()[0]
)
print(f"Largest difference between streaming and batch churn probability: {max_diff:.2e}")
assert max_diff < 1e-6, "Streaming scores differ from the batch scores - check the feature code."

display(
    spark.table(ALERTS_TABLE).join(truth, KEY_COL)
    .agg(
        F.count("*").alias("customers"),
        F.sum("alert").alias("alerts_raised"),
        F.sum(F.col(LABEL_COL) * F.col("alert")).alias("leavers_among_alerts"),
        F.sum(LABEL_COL).alias("all_leavers"),
    )
)

## Step 7 — What the retention team reads

**What we do:** summarise the alert table with Spark SQL, by country and by segment, and list the highest-risk customers.

**Why:** an alert is only useful if it says **who to call and what kind of customer they are**. The segment (GX_06) tells the team which offer fits; the country tells how urgent it is (Germany, GX_03).

In [ ]:
spark.table(ALERTS_TABLE).createOrReplaceTempView("alerts")

display(spark.sql('''
    SELECT Geography,
           COUNT(*)                     AS customers,
           SUM(alert)                   AS alerts,
           ROUND(AVG(alert) * 100, 1)   AS alert_pct
    FROM alerts
    GROUP BY Geography
    ORDER BY alert_pct DESC
'''))

display(spark.sql(f'''
    SELECT {SEGMENT_ID_COL}, {SEGMENT_NAME_COL},
           COUNT(*)                     AS customers,
           SUM(alert)                   AS alerts,
           ROUND(AVG(alert) * 100, 1)   AS alert_pct,
           ROUND(AVG(p_churn) * 100, 1) AS avg_risk_pct
    FROM alerts
    GROUP BY {SEGMENT_ID_COL}, {SEGMENT_NAME_COL}
    ORDER BY {SEGMENT_ID_COL}
'''))

display(spark.sql(f'''
    SELECT {KEY_COL}, Geography, Age, NumOfProducts, IsActiveMember, {SEGMENT_NAME_COL}, ROUND(p_churn, 3) AS p_churn
    FROM alerts
    WHERE alert = 1
    ORDER BY p_churn DESC
    LIMIT 10
'''))

## Summary

### What we did
- Simulated new customers arriving as files in a folder and processed them with **Spark Structured Streaming** (`readStream` -> `foreachBatch` -> Delta table), one micro-batch per file.
- Reused, without any change, the cleaning-feature code of GX_02, the segmentation model of GX_06 and the churn model and threshold of GX_04.
- Showed **exactly-once** processing: a second run with the same checkpoint read only the 5 new files and the table holds every customer exactly once.
- Showed that **streaming and batch scores are identical** (difference below 1e-6), so the streaming job is the validated model, not a copy of it.

### What the client gains
- **Velocity:** a new or changed customer can be scored in seconds. The bank does not wait for a monthly batch to see that a customer became high-risk.
- **One code base:** the model validated in GX_04 and tested in GX_05 is the model that scores new data.
- **Scale:** the micro-batch logic does not change if the folder is replaced by a message queue and the files by millions of events; Spark spreads each micro-batch over the cluster.

### Limits we state openly
- The arrival is simulated from a static file; the volume is tiny.
- Free Edition runs the query in *available now* mode; a production job would use a continuous trigger.
- The look-alike network (GX_07) is **not** part of the stream. Rebuilding the network for every small batch would cost a lot and would not change the scores.